In [11]:
import pandas as pd
import sqlite3

### Создадим подключение к БД с помощью sqlite3

In [12]:
conn = sqlite3.connect('../data/checking-logs.sqlite')

### Получим схему таблицы test

In [13]:
pd.read_sql('PRAGMA table_info(test)', conn)

,cid,name,type,notnull,dflt_value,pk
0,0,index,INTEGER,0,None,0
1,1,uid,TEXT,0,None,0
2,2,labname,TEXT,0,None,0
3,3,first_commit_ts,TIMESTAMP,0,None,0
4,4,first_view_ts,TIMESTAMP,0,None,0


### Получим первые 10 строк из таблицы test

In [14]:
pd.read_sql('SELECT * FROM test LIMIT 10', conn)

,index,uid,labname,first_commit_ts,first_view_ts
0,3,user_17,project1,2020-04-18 07:56:45,2020-04-18 10:56:55
1,4,user_30,laba04,2020-04-18 13:36:53,2020-04-17 22:46:26
2,7,user_30,laba04s,2020-04-18 14:51:37,2020-04-17 22:46:26
3,8,user_14,laba04,2020-04-18 15:14:00,2020-04-18 10:53:52
4,11,user_14,laba04s,2020-04-18 22:30:30,2020-04-18 10:53:52
5,18,user_19,laba04,2020-04-20 19:05:01,2020-04-21 20:30:38
6,19,user_25,laba04,2020-04-20 19:16:50,2020-05-09 23:54:54
7,20,user_21,laba04,2020-04-21 17:48:00,2020-04-22 22:40:36
8,21,user_30,project1,2020-04-22 12:36:24,2020-04-17 22:46:26
9,23,user_21,laba04s,2020-04-22 20:09:21,2020-04-22 22:40:36


### Найдите среди всех пользователей минимальное значение дельты между первым коммитом пользователя и крайним сроком выполнения соответствующей задачи, используя только один запрос

* сделайте это, объединив таблицу test с таблицей deadlines
* разница должна отображаться в часах
* не учитывайте 'project1', у него более длительные сроки выполнения и он будет выбросом
* значение должно храниться в фрейме данных df_min с соответствующим uid

In [15]:
query = """
SELECT uid, MIN(delta)
FROM (SELECT uid, cast((julianday(test.first_commit_ts) - julianday(datetime(dl.deadlines, 'unixepoch'))) * 24 AS INTEGER) AS delta
      FROM test
      LEFT JOIN deadlines AS dl ON dl.labs=test.labname
      WHERE NOT dl.labs = 'project1')
"""
df_min = pd.io.sql.read_sql(query, conn, index_col='uid')
df_min

,MIN(delta)
uid,
user_30,-202


### Теперь повторим то же самое, но для максимума

In [16]:
query = """
SELECT uid, MAX(delta)
FROM (SELECT uid, cast((julianday(test.first_commit_ts) - julianday(datetime(dl.deadlines, 'unixepoch'))) * 24 AS INTEGER) AS delta
      FROM test
      LEFT JOIN deadlines AS dl ON dl.labs=test.labname
      WHERE NOT dl.labs = 'project1')
"""
df_max = pd.io.sql.read_sql(query, conn, index_col='uid')
df_max

,MAX(delta)
uid,
user_25,-2


### Теперь для среднего

In [17]:
query = """
SELECT AVG(delta)
FROM (SELECT uid, cast((julianday(test.first_commit_ts) - julianday(datetime(dl.deadlines, 'unixepoch'))) * 24 AS INTEGER) AS delta
      FROM test
      LEFT JOIN deadlines AS dl ON dl.labs=test.labname
      WHERE NOT dl.labs = 'project1')
"""
df_avg = pd.io.sql.read_sql(query, conn)
df_avg

,AVG(delta)
0,-89.125


### Мы хотим проверить гипотезу о том, что у пользователей, которые просматривали ленту новостей всего несколько раз, разница между первым коммитом и дедлайном меньше. Для этого нужно вычислить коэффициент корреляции между количеством просмотров и разницей

* Используя только один запрос, создайте таблицу со столбцами: uid, avg_diff, pageviews
* uid — это идентификаторы пользователей, которые существуют в тесте
* avg_diff — это средняя разница между первым коммитом и сроком сдачи лабораторной работы для каждого пользователя
* pageviews — это количество просмотров ленты новостей для каждого пользователя
* не учитывайте лабораторную работу 'project1'
* сохраните её в фреймворке views_diff
* используйте метод Pandas corr() для вычисления коэффициента корреляции между количеством просмотров и разницей

In [19]:
query = """
SELECT uid,
       AVG(delta) AS avg_diff,
       COUNT(uid) AS pageviews
FROM (SELECT uid, cast((julianday(test.first_commit_ts) - julianday(datetime(dl.deadlines, 'unixepoch'))) * 24 AS INTEGER) AS delta
      FROM test
      LEFT JOIN deadlines AS dl ON dl.labs=test.labname
      WHERE NOT dl.labs = 'project1')
GROUP BY uid
"""
views_diff = pd.io.sql.read_sql(query, conn, index_col='uid')
views_diff.corr(method='pearson')

,avg_diff,pageviews
avg_diff,1.000000,0.117685
pageviews,0.117685,1.000000


### Закроем соединение с базой данных

In [20]:
conn.close()